In [2]:
import torch
from unsloth import FastLanguageModel
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from pathlib import Path

persist_directory = "/workspaces/Arch_PC_Assistent/embed_model"
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma(embedding_function=embeddings,
                     persist_directory=persist_directory)


path_model_v1 = "/workspaces/Arch_PC_Assistent/outputs/SFT/arch_assistant_lora_1"
max_seq_length = 4096 
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = path_model_v1,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)


test_frage = "My Waybar is crashing on startup. How can I debug or fix it?"

docs = vectorstore.similarity_search(test_frage, k=3)
rag_context = "\n\n".join([f"--- Chunk {i+1} ({d.metadata.get('topic')}) ---\n{d.page_content}" for i, d in enumerate(docs)])

system_prompt = f"""You are ArchAgent, an expert Arch Linux assistant. 
Use ONLY the following context to answer the user's question. If the answer is not in the context, say so.
Your output MUST follow this format:
<think>
[Your reasoning here]
</think>
<answer>
[Your final concise answer here]
</answer>
CONTEXT:
{rag_context}"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_frage}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")


outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    use_cache=True,
    do_sample=True,      # WICHTIG für Varianz!
    temperature=0.7,     # Macht das Modell kreativer in der Lösungsfindung
    top_p=0.9,           # Schneidet komplett unlogische Wörter ab
    num_return_sequences=6 # Erzeugt 6 Antworten auf einmal
)

# Ergebnisse anzeigen
decoded_outputs = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for i, answer in enumerate(decoded_outputs):
    print(f"{"="*40}")
    print(f"🌟 ANTWORT {i+1} 🌟")
    print(f"{"="*40}")
    print(answer)
    print("\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4080 SUPER. Num GPUs = 1. Max memory: 15.55 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/workspaces/Arch_PC_Assistent/.venv/lib/python3.13/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/workspaces/Arch_PC_Assistent/.venv/lib/python3.13/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.1

🌟 ANTWORT 1 🌟
Waybar crashes can often be due to missing required libraries, incorrect configuration syntax, or conflicting plugins. Here’s a systematic approach to diagnose and resolve:

### 1. Check for Missing Dependencies
Ensure all dependencies are installed:
```bash
sudo pacman -S waybar waybar-plugins
```
If you use a custom build, check `waybar` and plugin logs.

### 2. Inspect Logs
Waybar writes detailed logs to `~/.cache/waybar/waybar.log`. Open this file to see the exact error message. Look for lines like:
```
[error] ... failed to load plugin 'plugin-name'
```

### 3. Validate Configuration
Syntax errors in `~/.config/waybar/config.jsonc` will cause immediate failure. Use a linter:
```bash
waybar-config --validate ~/.config/waybar/config.jsonc
```
If no validator exists, manually check each line for correctness.

### 4. Test with a Minimal Config
Create a minimal config file to isolate the issue:
```json
{
    "main": {
        "monitor": "all",
        "workspaces": {
    